In [1]:
from transformers import pipeline

pipe = pipeline("text-generation", model="google/gemma-3-270m-it", device='mps')
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

Device set to use mps


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': 'I am Gemma, an open-weights AI assistant. I am a large language model created by the Gemma team at Google DeepMind.\n'}]}]

In [2]:
from inference_tool import GemmaModel, Tokenizer, generate, format_messages

model = GemmaModel("google/gemma-3-270m-it")
tokenizer = Tokenizer("google/gemma-3-270m-it")

prompt = format_messages([{"role": "user", "content": "Who are you?"}])

tokens = tokenizer.encode(prompt)
output = generate(model, tokenizer, tokens, max_new_tokens=30, temperature=0.0)
print(tokenizer.decode(output))

user
Who are you?
model
I am Gemma, an open-weights AI assistant. I am a large language model created by the Gemma team at Google DeepMind.



In [3]:
# ============================================================================
# STEP-BY-STEP: How text generation works in a Gemma model
# ============================================================================

import numpy as np
from inference_tool import GemmaModel, Tokenizer, format_messages

# --- Step 0: Load model and tokenizer ---
# The model loads pre-trained weights from Hugging Face:
#   - Token embeddings: maps token IDs to dense vectors (vocab_size x hidden_size)
#   - Transformer layers: attention + MLP blocks that process sequences
#   - LM head: projects hidden states back to vocabulary logits
model = GemmaModel("google/gemma-3-270m-it")
tokenizer = Tokenizer("google/gemma-3-270m-it")

In [4]:
# --- Step 1: Format and tokenize the prompt ---
# Chat models require special formatting with role markers and turn tokens
prompt = format_messages([{"role": "user", "content": "Who are you?"}])
print("Formatted prompt:")
print(repr(prompt))

# Tokenize: convert text to token IDs
prompt_tokens = tokenizer.encode(prompt)
print(f"\nPrompt tokens: {prompt_tokens}")
print(f"Token count: {len(prompt_tokens)}")

Formatted prompt:
'<start_of_turn>user\nWho are you?<end_of_turn>\n<start_of_turn>model\n'

Prompt tokens: [105, 2364, 107, 15938, 659, 611, 236881, 106, 107, 105, 4368, 107]
Token count: 12


In [5]:
# --- Step 2: Add BOS token if needed ---
# Gemma models require a Beginning-of-Sequence token at the start
bos_token_id = tokenizer.bos_token_id
if prompt_tokens[0] != bos_token_id:
    prompt_tokens = [bos_token_id] + list(prompt_tokens)
    print(f"\nAdded BOS token: {prompt_tokens}")

# Initialize output sequence with prompt tokens
output_tokens = list(prompt_tokens)


Added BOS token: [2, 105, 2364, 107, 15938, 659, 611, 236881, 106, 107, 105, 4368, 107]


In [6]:
# ============================================================================
# PREFILL PHASE: Process the entire prompt in one forward pass
# ============================================================================

# --- Step 3: Convert tokens to input tensor ---
# Shape: (batch_size=1, seq_len=N)
token_ids = np.array([prompt_tokens], dtype=np.int32)
print(f"\nInput shape: {token_ids.shape}")


Input shape: (1, 13)


In [7]:
# --- Step 4: Forward pass overview ---
# The forward pass transforms token IDs into logits (probability scores over vocabulary).
# We'll break this down step by step in the following cells.
#
# High-level flow:
#   a) Embedding lookup + scaling
#   b) Process through N transformer layers (Gemma-3-270M has 18 layers)
#      - Each layer: Attention + MLP with residual connections
#   c) Final RMSNorm
#   d) LM head projection to vocabulary logits
#
# Let's trace through each step manually!

print(f"Model architecture:")
print(f"  - Vocabulary size: {model.vocab_size:,}")
print(f"  - Hidden size: {model.hidden_size}")
print(f"  - Number of layers: {model.num_layers}")
print(f"  - Input shape: {token_ids.shape}")

Model architecture:
  - Vocabulary size: 262,144
  - Hidden size: 640
  - Number of layers: 18
  - Input shape: (1, 13)


In [8]:
# --- Step 4a: Embedding lookup ---
# Token IDs are mapped to dense vectors via a lookup table.
# Shape: (batch=1, seq_len) → (batch=1, seq_len, hidden_size)

# The embedding table is a matrix of shape (vocab_size, hidden_size)
print(f"Embedding table shape: {model.embed_tokens.shape}")
print(f"  - vocab_size={model.vocab_size}, hidden_size={model.hidden_size}")

# Look up embeddings for our tokens
embeddings_raw = model.embed_tokens[token_ids]
print(f"\nAfter lookup:")
print(f"  Input token_ids shape: {token_ids.shape}")
print(f"  Output embeddings shape: {embeddings_raw.shape}")

Embedding table shape: (262144, 640)
  - vocab_size=262144, hidden_size=640

After lookup:
  Input token_ids shape: (1, 13)
  Output embeddings shape: (1, 13, 640)


In [9]:
# --- Step 4b: Gemma embedding scaling ---
# Gemma scales embeddings by sqrt(hidden_size) for better gradient flow.
# This is a model-specific design choice.

scale_factor = np.sqrt(model.hidden_size)
print(f"Scaling factor: √{model.hidden_size} = {scale_factor:.2f}")

hidden_states = embeddings_raw * scale_factor
hidden_states = hidden_states.astype(np.float32)

print(f"\nEmbedding statistics before scaling:")
print(f"  Mean: {embeddings_raw.mean():.4f}, Std: {embeddings_raw.std():.4f}")
print(f"\nEmbedding statistics after scaling:")
print(f"  Mean: {hidden_states.mean():.4f}, Std: {hidden_states.std():.4f}")
print(f"\nHidden states shape: {hidden_states.shape}")

Scaling factor: √640 = 25.30

Embedding statistics before scaling:
  Mean: -0.0006, Std: 0.0409

Embedding statistics after scaling:
  Mean: -0.0150, Std: 1.0355

Hidden states shape: (1, 13, 640)


In [10]:
# --- Step 4c: Process through Transformer layers ---
# Gemma-3-270M has 18 transformer layers. Each layer has:
#   1. Input LayerNorm → Self-Attention → Post-Attention LayerNorm → Residual Add
#   2. Pre-FF LayerNorm → MLP → Post-FF LayerNorm → Residual Add
#
# We'll trace through ONE layer in detail, then run all layers.

print(f"Number of transformer layers: {model.num_layers}")
print("\nEach TransformerBlock contains:")
print("  • input_layernorm (RMSNorm)")
print("  • self_attn (Multi-Head Attention with GQA)")
print("  • post_attention_layernorm (RMSNorm)")
print("  • pre_feedforward_layernorm (RMSNorm)")
print("  • mlp (GEGLU feed-forward)")
print("  • post_feedforward_layernorm (RMSNorm)")

# Initialize KV cache for all layers
kv_cache = [None] * model.num_layers

Number of transformer layers: 18

Each TransformerBlock contains:
  • input_layernorm (RMSNorm)
  • self_attn (Multi-Head Attention with GQA)
  • post_attention_layernorm (RMSNorm)
  • pre_feedforward_layernorm (RMSNorm)
  • mlp (GEGLU feed-forward)
  • post_feedforward_layernorm (RMSNorm)


In [11]:
# --- Step 4c.1: Inside a Transformer layer - Input RMSNorm ---
# Let's trace through Layer 0 step by step.
#
# RMSNorm (Root Mean Square Normalization):
#   output = (x / sqrt(mean(x²) + eps)) * (1 + weight)
#
# Unlike LayerNorm, RMSNorm doesn't center the data (no mean subtraction).

layer_0 = model.layers[0]
x = hidden_states.copy()  # Save input for residual connection

# Apply input layer normalization
x_norm = layer_0.input_layernorm(x)

print(f"Input to layer 0:")
print(f"  Shape: {x.shape}")
print(f"  Mean: {x.mean():.4f}, Std: {x.std():.4f}")
print(f"\nAfter input_layernorm:")
print(f"  Shape: {x_norm.shape}")
print(f"  Mean: {x_norm.mean():.4f}, Std: {x_norm.std():.4f}")

Input to layer 0:
  Shape: (1, 13, 640)
  Mean: -0.0150, Std: 1.0355

After input_layernorm:
  Shape: (1, 13, 640)
  Mean: -0.5102, Std: 17.0509


In [12]:
# --- Step 4c.2: Self-Attention - Q, K, V projections ---
# Multi-head attention first projects the input into Query, Key, and Value tensors.
# Gemma uses Grouped Query Attention (GQA): fewer K,V heads than Q heads to save memory.

attn = layer_0.self_attn
batch_size, seq_len, _ = x_norm.shape

print(f"Attention configuration:")
print(f"  num_query_heads: {attn.num_heads}")
print(f"  num_kv_heads: {attn.num_kv_heads}")
print(f"  head_dim: {attn.head_dim}")
print(f"  heads_per_kv: {attn.num_heads_per_kv} (GQA ratio)")

# Project to Q, K, V
q = x_norm @ attn.q_proj  # (batch, seq, num_heads * head_dim)
k = x_norm @ attn.k_proj  # (batch, seq, num_kv_heads * head_dim)
v = x_norm @ attn.v_proj  # (batch, seq, num_kv_heads * head_dim)

print(f"\nProjection shapes:")
print(f"  Q: {q.shape} → {attn.num_heads} heads × {attn.head_dim} dim")
print(f"  K: {k.shape} → {attn.num_kv_heads} heads × {attn.head_dim} dim")
print(f"  V: {v.shape} → {attn.num_kv_heads} heads × {attn.head_dim} dim")

Attention configuration:
  num_query_heads: 4
  num_kv_heads: 1
  head_dim: 256
  heads_per_kv: 4 (GQA ratio)

Projection shapes:
  Q: (1, 13, 1024) → 4 heads × 256 dim
  K: (1, 13, 256) → 1 heads × 256 dim
  V: (1, 13, 256) → 1 heads × 256 dim


In [13]:
# --- Step 4c.3: Reshape to separate heads ---
# Reshape from (batch, seq, heads*dim) to (batch, heads, seq, dim)
# This prepares tensors for parallel attention computation across heads.

q_heads = q.reshape(batch_size, seq_len, attn.num_heads, attn.head_dim)
k_heads = k.reshape(batch_size, seq_len, attn.num_kv_heads, attn.head_dim)
v_heads = v.reshape(batch_size, seq_len, attn.num_kv_heads, attn.head_dim)

# Transpose to (batch, heads, seq, dim)
q_heads = q_heads.transpose(0, 2, 1, 3)
k_heads = k_heads.transpose(0, 2, 1, 3)
v_heads = v_heads.transpose(0, 2, 1, 3)

print(f"After reshape and transpose:")
print(f"  Q: {q_heads.shape}  (batch, heads, seq, head_dim)")
print(f"  K: {k_heads.shape}  (batch, kv_heads, seq, head_dim)")
print(f"  V: {v_heads.shape}  (batch, kv_heads, seq, head_dim)")

After reshape and transpose:
  Q: (1, 4, 13, 256)  (batch, heads, seq, head_dim)
  K: (1, 1, 13, 256)  (batch, kv_heads, seq, head_dim)
  V: (1, 1, 13, 256)  (batch, kv_heads, seq, head_dim)


In [14]:
# --- Step 4c.4: Q/K Normalization (Gemma3-specific) ---
# Gemma3 applies RMSNorm to Q and K after projection but before RoPE.
# This helps stabilize attention scores.

def apply_qk_norm(tensor, weight, eps=1e-6):
    """RMSNorm with (1 + weight) formulation."""
    variance = np.mean(tensor**2, axis=-1, keepdims=True)
    return (tensor / np.sqrt(variance + eps)) * (1.0 + weight)

q_normed = apply_qk_norm(q_heads, attn.q_norm_weight)
k_normed = apply_qk_norm(k_heads, attn.k_norm_weight)

print(f"Q/K after normalization:")
print(f"  Q norm weight shape: {attn.q_norm_weight.shape}")
print(f"  Q stats - Mean: {q_normed.mean():.4f}, Std: {q_normed.std():.4f}")
print(f"  K stats - Mean: {k_normed.mean():.4f}, Std: {k_normed.std():.4f}")

Q/K after normalization:
  Q norm weight shape: (256,)
  Q stats - Mean: 0.0663, Std: 1.2628
  K stats - Mean: -0.0215, Std: 1.6105


In [15]:
# --- Step 4c.5: Rotary Position Embeddings (RoPE) ---
# RoPE encodes position by rotating pairs of dimensions in Q and K.
# This allows the model to understand token positions without explicit position embeddings.
#
# For each pair of dimensions (x, y) at position p:
#   x' = x * cos(θp) - y * sin(θp)
#   y' = x * sin(θp) + y * cos(θp)
#
# θ = base^(-2i/d) where i is the dimension pair index

from inference_tool.layers import apply_rope

positions = np.arange(0, seq_len)  # [0, 1, 2, ..., seq_len-1]
print(f"Position indices: {positions}")
print(f"RoPE base frequency: {attn.rope_base}")

q_rope, k_rope = apply_rope(q_normed, k_normed, positions, attn.head_dim, attn.rope_base)

print(f"\nAfter RoPE:")
print(f"  Q shape: {q_rope.shape}")
print(f"  K shape: {k_rope.shape}")
print(f"\nRoPE encodes position information via rotation - same content at")
print(f"different positions will have different Q/K representations.")

Position indices: [ 0  1  2  3  4  5  6  7  8  9 10 11 12]
RoPE base frequency: 10000.0

After RoPE:
  Q shape: (1, 4, 13, 256)
  K shape: (1, 1, 13, 256)

RoPE encodes position information via rotation - same content at
different positions will have different Q/K representations.


In [16]:
# --- Step 4c.6: GQA - Repeat K,V heads to match Q heads ---
# Grouped Query Attention: Multiple Q heads share the same K,V heads.
# We repeat K,V to match the number of Q heads for the attention computation.

if attn.num_heads_per_kv > 1:
    k_expanded = np.repeat(k_rope, attn.num_heads_per_kv, axis=1)
    v_expanded = np.repeat(v_heads, attn.num_heads_per_kv, axis=1)
else:
    k_expanded = k_rope
    v_expanded = v_heads

print(f"GQA expansion (each KV head serves {attn.num_heads_per_kv} Q heads):")
print(f"  K: {k_rope.shape} → {k_expanded.shape}")
print(f"  V: {v_heads.shape} → {v_expanded.shape}")
print(f"  Q: {q_rope.shape} (unchanged)")

GQA expansion (each KV head serves 4 Q heads):
  K: (1, 1, 13, 256) → (1, 4, 13, 256)
  V: (1, 1, 13, 256) → (1, 4, 13, 256)
  Q: (1, 4, 13, 256) (unchanged)


In [17]:
# --- Step 4c.7: Compute attention scores ---
# Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d)) @ V
#
# Each query asks: "What should I attend to?"
# The dot product Q @ K^T measures similarity between query and key positions.

# Q @ K^T: (batch, heads, seq_q, dim) @ (batch, heads, dim, seq_k) → (batch, heads, seq_q, seq_k)
attn_scores = q_rope @ k_expanded.transpose(0, 1, 3, 2)

# Scale by sqrt(head_dim) or custom scalar
attn_scores = attn_scores * attn.scale

print(f"Attention scores shape: {attn_scores.shape}")
print(f"  Each position attends to all other positions")
print(f"  Scale factor: {attn.scale:.4f}")
print(f"\nScores for head 0, position 0 attending to all positions:")
print(f"  {attn_scores[0, 0, 0, :5]}... (first 5 positions)")

Attention scores shape: (1, 4, 13, 13)
  Each position attends to all other positions
  Scale factor: 0.0625

Scores for head 0, position 0 attending to all positions:
  [5.66555691 0.11279435 0.11287916 5.69199705 3.97571731]... (first 5 positions)


In [18]:
# --- Step 4c.8: Apply causal mask ---
# For autoregressive generation, each position can only attend to previous positions.
# We mask future positions with -inf (becomes 0 after softmax).

def create_causal_mask(seq_len):
    """Create lower triangular mask: position i can only see positions 0..i"""
    mask = np.zeros((seq_len, seq_len), dtype=np.float32)
    for i in range(seq_len):
        mask[i, i+1:] = -np.inf  # Mask future positions
    return mask.reshape(1, 1, seq_len, seq_len)

causal_mask = create_causal_mask(seq_len)
attn_scores_masked = attn_scores + causal_mask

print(f"Causal mask (visualized, 0=attend, -inf=masked):")
mask_viz = np.where(causal_mask[0, 0] == 0, "○", "✗")
for i in range(min(6, seq_len)):
    print(f"  pos {i}: {' '.join(mask_viz[i, :min(6, seq_len)])}")
print(f"\nMasked attention scores for head 0, last position:")
print(f"  {attn_scores_masked[0, 0, -1, :]}")

Causal mask (visualized, 0=attend, -inf=masked):
  pos 0: ○ ✗ ✗ ✗ ✗ ✗
  pos 1: ○ ○ ✗ ✗ ✗ ✗
  pos 2: ○ ○ ○ ✗ ✗ ✗
  pos 3: ○ ○ ○ ○ ✗ ✗
  pos 4: ○ ○ ○ ○ ○ ✗
  pos 5: ○ ○ ○ ○ ○ ○

Masked attention scores for head 0, last position:
  [ 1.89207625 -0.24761651  0.16319938  2.86084223  0.63307005  0.68891275
  0.49958977  1.42210603  2.44440508  3.025841   -0.42168379  1.10018122
  3.00678849]


In [19]:
# --- Step 4c.9: Softmax → attention weights ---
# Softmax converts scores to probabilities (sum to 1 across attended positions).
# The -inf masked positions become 0 probability.

from inference_tool.layers import softmax

attn_weights = softmax(attn_scores_masked, axis=-1)

print(f"Attention weights shape: {attn_weights.shape}")
print(f"\nHead 0, last position attention weights (should sum to 1):")
weights_last = attn_weights[0, 0, -1, :]
print(f"  Sum: {weights_last.sum():.4f}")
print(f"  Weights: {weights_last}")
print(f"\nThe last position attends to all previous positions with these weights.")

Attention weights shape: (1, 4, 13, 13)

Head 0, last position attention weights (should sum to 1):
  Sum: 1.0000
  Weights: [0.07229239 0.00850816 0.01283072 0.19046812 0.02052642 0.02170528
 0.01796153 0.04518425 0.12559315 0.22463652 0.0071489  0.03274741
 0.22039714]

The last position attends to all previous positions with these weights.


In [20]:
# --- Step 4c.10: Compute attention output ---
# Weighted sum of values: attn_weights @ V
# Each position's output is a weighted combination of all attended values.

# (batch, heads, seq_q, seq_k) @ (batch, heads, seq_k, dim) → (batch, heads, seq_q, dim)
attn_output = attn_weights @ v_expanded

print(f"Attention output shape: {attn_output.shape}")

# Reshape back: (batch, heads, seq, dim) → (batch, seq, heads, dim) → (batch, seq, heads*dim)
attn_output = attn_output.transpose(0, 2, 1, 3)
attn_output = attn_output.reshape(batch_size, seq_len, -1)

print(f"After reshape: {attn_output.shape}")

Attention output shape: (1, 4, 13, 256)
After reshape: (1, 13, 1024)


In [21]:
# --- Step 4c.11: Output projection + residual connection ---
# Project attention output back to hidden_size, then add residual.
# Gemma3 has a post-attention LayerNorm BEFORE the residual add.

# Output projection
attn_projected = attn_output @ attn.o_proj
print(f"After output projection: {attn_projected.shape}")

# Post-attention LayerNorm (Gemma3 specific)
attn_normed = layer_0.post_attention_layernorm(attn_projected)

# Residual connection: add original input back
residual = x  # We saved this at the start of the layer
hidden_after_attn = residual + attn_normed

print(f"After residual connection: {hidden_after_attn.shape}")
print(f"\nResidual connections prevent vanishing gradients and allow")
print(f"the model to learn 'what to add' rather than 'what to output'.")

After output projection: (1, 13, 640)
After residual connection: (1, 13, 640)

Residual connections prevent vanishing gradients and allow
the model to learn 'what to add' rather than 'what to output'.


In [22]:
# --- Step 4c.12: MLP (Feed-Forward Network) ---
# The MLP expands the hidden dimension, applies non-linearity, then projects back.
# Gemma uses GEGLU: gate = GELU(x @ W_gate), up = x @ W_up, out = (gate * up) @ W_down

from inference_tool.layers import gelu

mlp = layer_0.mlp
residual_mlp = hidden_after_attn  # Save for residual

# Pre-feedforward LayerNorm
ff_input = layer_0.pre_feedforward_layernorm(hidden_after_attn)

print(f"MLP dimensions:")
print(f"  Hidden size: {mlp.hidden_size}")
print(f"  Intermediate size: {mlp.intermediate_size} ({mlp.intermediate_size/mlp.hidden_size:.1f}x expansion)")

# Gate projection with GELU activation
gate = gelu(ff_input @ mlp.gate_proj)

# Up projection (no activation)
up = ff_input @ mlp.up_proj

# Gating: element-wise multiplication
hidden = gate * up

# Down projection back to hidden_size
mlp_output = hidden @ mlp.down_proj

print(f"\nMLP forward pass:")
print(f"  Input: {ff_input.shape}")
print(f"  Gate/Up: {gate.shape}")
print(f"  Output: {mlp_output.shape}")

MLP dimensions:
  Hidden size: 640
  Intermediate size: 2048 (3.2x expansion)

MLP forward pass:
  Input: (1, 13, 640)
  Gate/Up: (1, 13, 2048)
  Output: (1, 13, 640)


In [23]:
# --- Step 4c.13: Post-MLP LayerNorm + residual ---
# Final step of the transformer layer: normalize and add residual.

# Post-feedforward LayerNorm (Gemma3 specific)
mlp_normed = layer_0.post_feedforward_layernorm(mlp_output)

# Residual connection
layer_0_output = residual_mlp + mlp_normed

print(f"Layer 0 complete!")
print(f"  Input shape: {x.shape}")
print(f"  Output shape: {layer_0_output.shape}")
print(f"\nThis output becomes the input to Layer 1, and so on for all 18 layers.")

Layer 0 complete!
  Input shape: (1, 13, 640)
  Output shape: (1, 13, 640)

This output becomes the input to Layer 1, and so on for all 18 layers.


In [24]:
# --- Step 4d: Run all transformer layers ---
# Now let's run through all 18 layers (using the model's built-in method for efficiency).
# Each layer builds on the previous, refining the representation.

print("Processing through all transformer layers...")
hidden_states_all = hidden_states.copy()  # Start fresh from embeddings

new_kv_cache = []
for i, layer in enumerate(model.layers):
    hidden_states_all, layer_kv_cache = layer(
        hidden_states_all,
        start_pos=0,
        kv_cache=kv_cache[i],
    )
    new_kv_cache.append(layer_kv_cache)
    if i % 6 == 0 or i == model.num_layers - 1:
        print(f"  Layer {i:2d}: mean={hidden_states_all.mean():.4f}, std={hidden_states_all.std():.4f}")

kv_cache = new_kv_cache
print(f"\nAll {model.num_layers} layers complete!")
print(f"Final hidden states shape: {hidden_states_all.shape}")

Processing through all transformer layers...
  Layer  0: mean=0.3691, std=32.5918
  Layer  6: mean=15.4194, std=607.2468
  Layer 12: mean=28.9765, std=1183.6096
  Layer 17: mean=18.7894, std=1060.4789

All 18 layers complete!
Final hidden states shape: (1, 13, 640)


In [25]:
# --- Step 4e: Final RMSNorm ---
# One last normalization before projecting to vocabulary.

hidden_states_normed = model.norm(hidden_states_all)

print(f"After final RMSNorm:")
print(f"  Shape: {hidden_states_normed.shape}")
print(f"  Mean: {hidden_states_normed.mean():.4f}, Std: {hidden_states_normed.std():.4f}")

After final RMSNorm:
  Shape: (1, 13, 640)
  Mean: -0.2604, Std: 9.5632


In [26]:
# --- Step 4f: LM Head - Project to vocabulary ---
# The language model head projects hidden states to vocabulary logits.
# Each position gets a score for every token in the vocabulary.

print(f"LM head weight shape: {model.lm_head.shape}")
print(f"  (hidden_size, vocab_size) = ({model.hidden_size}, {model.vocab_size})")

logits = hidden_states_normed @ model.lm_head

print(f"\nOutput logits shape: {logits.shape}")
print(f"  (batch_size, seq_len, vocab_size)")
print(f"\nEach of the {logits.shape[1]} positions has {logits.shape[2]:,} logits,")
print(f"one for every token in the vocabulary.")
print(f"\n✅ Forward pass complete! We now have logits to sample from.")

LM head weight shape: (640, 262144)
  (hidden_size, vocab_size) = (640, 262144)

Output logits shape: (1, 13, 262144)
  (batch_size, seq_len, vocab_size)

Each of the 13 positions has 262,144 logits,
one for every token in the vocabulary.

✅ Forward pass complete! We now have logits to sample from.


In [27]:
# --- Step 5: Extract logits for the last position ---
# We only care about the last token's predictions (what comes next?)
next_token_logits = logits[0, -1, :].copy()
print(f"\nLast position logits shape: {next_token_logits.shape}")  # (vocab_size,)

# Show top 5 candidate tokens
def show_top_tokens(logits, tokenizer, k=5):
    top_indices = np.argsort(logits)[-k:][::-1]
    print("Top predicted tokens:")
    for idx in top_indices:
        token_id = int(idx)  # Convert numpy int to Python int for tokenizer
        print(f"  {token_id}: '{tokenizer.decode([token_id])}' (logit: {logits[idx]:.2f})")

show_top_tokens(next_token_logits, tokenizer)


Last position logits shape: (262144,)
Top predicted tokens:
  236777: 'I' (logit: 23.42)
  106: '' (logit: 13.36)
  2205: 'As' (logit: 13.35)
  40281: 'मैं' (logit: 11.58)
  9259: 'Hello' (logit: 11.13)


In [28]:
# --- Step 6: Sample the next token (greedy with temperature=0) ---
# Temperature=0 means we just take argmax (most likely token)
def sample_greedy(logits):
    return int(np.argmax(logits))

first_new_token = sample_greedy(next_token_logits)
output_tokens.append(first_new_token)
print(f"\n🎯 Sampled token: {first_new_token} = '{tokenizer.decode([first_new_token])}'");


🎯 Sampled token: 236777 = 'I'


In [29]:
# ============================================================================
# DECODE PHASE: Generate tokens one at a time using KV cache
# ============================================================================

print("\n📍 DECODE: Generating tokens one by one...")

# Define stop tokens
stop_tokens = {tokenizer.eos_token_id, 106}  # 106 = <end_of_turn>
max_new_tokens = 30

for step in range(max_new_tokens - 1):
    # --- Step 7: Prepare input for next token ---
    # Only pass the LAST generated token (not the whole sequence!)
    # The KV cache stores previous tokens' key/value pairs
    current_token = output_tokens[-1]
    token_ids = np.array([[current_token]], dtype=np.int32)
    
    # Position offset tells the model where we are in the sequence
    # (used for RoPE positional encoding)
    position_offset = len(output_tokens) - 1
    
    # --- Step 8: Forward pass with KV cache ---
    # This is FAST because:
    #   - We only process 1 token instead of the whole sequence
    #   - Attention uses cached keys/values from previous positions
    #   - Only need to compute attention for new token against all previous
    logits, kv_cache = model.forward(
        token_ids, 
        position_offset=position_offset, 
        kv_cache=kv_cache
    )
    
    # --- Step 9: Sample next token ---
    next_token_logits = logits[0, -1, :].copy()
    next_token = sample_greedy(next_token_logits)
    output_tokens.append(next_token)
    
    # Show progress
    decoded_so_far = tokenizer.decode([next_token])
    print(f"  Step {step+1}: token={next_token:>5}, text='{decoded_so_far}'")
    
    # --- Step 10: Check for stop condition ---
    if next_token in stop_tokens:
        print(f"\n🛑 Stop token reached!")
        break


📍 DECODE: Generating tokens one by one...
  Step 1: token= 1006, text=' am'
  Step 2: token=147224, text=' Gemma'
  Step 3: token=236764, text=','
  Step 4: token=  614, text=' an'
  Step 5: token= 1932, text=' open'
  Step 6: token=236772, text='-'
  Step 7: token=38357, text='weights'
  Step 8: token=12498, text=' AI'
  Step 9: token=16326, text=' assistant'
  Step 10: token=236761, text='.'
  Step 11: token=  564, text=' I'
  Step 12: token= 1006, text=' am'
  Step 13: token=  496, text=' a'
  Step 14: token= 2455, text=' large'
  Step 15: token= 5192, text=' language'
  Step 16: token= 2028, text=' model'
  Step 17: token= 4464, text=' created'
  Step 18: token=  684, text=' by'
  Step 19: token=  506, text=' the'
  Step 20: token=147224, text=' Gemma'
  Step 21: token= 2434, text=' team'
  Step 22: token=  657, text=' at'
  Step 23: token= 6475, text=' Google'
  Step 24: token=22267, text=' Deep'
  Step 25: token=65153, text='Mind'
  Step 26: token=236761, text='.'
  Step 27: tok

In [30]:
# ============================================================================
# FINAL OUTPUT
# ============================================================================
print("\n" + "="*60)
print("FINAL GENERATED TEXT:")
print("="*60)
print(tokenizer.decode(output_tokens))


FINAL GENERATED TEXT:
user
Who are you?
model
I am Gemma, an open-weights AI assistant. I am a large language model created by the Gemma team at Google DeepMind.

